In [ ]:
import numpy as np
import scipy.signal as si
import matplotlib.pyplot as plt

from scipy.constants import c

In [ ]:
from lisainstrument import Instrument

## Exercise 1: More noises

1. Simulate one day of LISA data at a sampling rate of 4 Hz including OMS (use default levels in all interferometers, see `oms_asds` parameter in [docs](https://lisa-simulation.pages.in2p3.fr/instrument/instrument.html)) and test-mass noise. For this experiment choose static arms by setting `orbits="static"` (no orbit file required). Export the result into a numpy array by calling `results = instru.export_numpy_core()` instead of writing it to disk. The individual interferometric measurements can be accessed via e.g. `results.mosa_data.sci_carrier_fluctuations["12"]`. Hint: You can look up all data set names in `results.mosa_data.dataset_names()`. Have a look at what is available. You do not need to understand all of them, but try to recognise the SCI/REF/TMI carrier measurements on each MOSA.
2. Compute the amplitude spectral densities (ASDs) of the SCI/REF/TMI  beatnotes using the `welch` function and compare the results to analytic curves. Is the model able to fully explain the data? Make sure to choose the correct `level` parameter for the OMS noise in the SCI, TMI and REF (`oms_asds[0]`, `oms_asds[2]` and `oms_asds[4]`).

In [ ]:
nu0 = 281600000000000.0 # central laser frequency in Hz
lambda0 = c / nu0

def asd_oms(f, level=6.35e-12, f_knee=2e-3):
    level_nu = level * (2 * np.pi * f) / lambda0 # convert from displacement units to frequency units
    return level_nu * np.sqrt((1 + (f_knee / f)**4))
    
def asd_tm(f, level=2.4e-15, f_knee=4e-4, f_break=8e-3):
    level_nu = level / (2 * np.pi * f) / lambda0 # convert from acceleration units to frequency units
    return level_nu * np.sqrt((1 + (f_knee / f)**2) * (1 + (f / f_break)**4))

In [ ]:
fs = 4 # sampling rate of the data (Hz)
size = (3600 * 24) * fs # one day of simulation in number of samples
oms_asds = (6.35e-12, 1.25e-11, 1.42e-12, 3.38e-12, 3.32e-12, 7.9e-12)

instru = Instrument(
    size=size,
    dt=1 / fs,
    orbits="static",
    lock="six",
    oms_asds=oms_asds,
    testmass_asds=2.4e-15, # TM noise level in m / s^2 / rtHz
)

instru.disable_all_noises(excluding=["oms", "test-mass"])

results = instru.export_numpy_core()

In [ ]:
# options for spectral estimation
BUFFER = 1000
welch_kwargs = dict(window=("kaiser", si.kaiser_beta(240)), nperseg=50_000, fs=fs, detrend=None)

In [ ]:
results.mosa_data.dataset_names()

In [ ]:
freqs, psd_sci = si.welch(results.mosa_data.sci_carrier_fluctuations["12"], **welch_kwargs)
freqs, psd_ref = si.welch(results.mosa_data.ref_carrier_fluctuations["12"], **welch_kwargs)
freqs, psd_tmi = si.welch(results.mosa_data.tmi_carrier_fluctuations["12"], **welch_kwargs)

plt.loglog(freqs, np.sqrt(psd_sci), label="sci data")
plt.loglog(freqs[1:], asd_oms(freqs[1:], level=oms_asds[0]), label="sci model")

plt.loglog(freqs, np.sqrt(psd_ref), label="ref data")
plt.loglog(freqs[1:], asd_oms(freqs[1:], level=oms_asds[4]), label="ref model")

plt.loglog(freqs, np.sqrt(psd_tmi), label="tmi data")
plt.loglog(freqs[1:], np.sqrt(4 * asd_tm(freqs[1:])**2 + asd_oms(freqs[1:], level=oms_asds[2])**2), label="tmi model")


plt.xlabel("Frequency (Hz)")
plt.ylabel("ASD (Hz/rtHz)")
plt.legend()

## Exercise 2: Laser noise

So far the models explained the data. But in the real instrument the lasers are not perfectly stable, and laser frequency noise $p_{ij}$ enters every measurement.

1. Include laser frequency noise in the simulation (add `"laser"` to the list of noises to include) and repeat the experiment from above. What can you observe in the estimated ASDs of the individual interferometric measurements?
2. Look at the equations again: $\mathrm{ref}_{12}$ and $\mathrm{tmi}_{12}$ contain the same laser noise $p_{13}-p_{12}$, so their difference should be free of it. Check whether the difference of $\mathrm{ref}_{12}$ and $\mathrm{tmi}_{12}$ is again free of laser frequency noise. (Even if this is not really helpful for the science as the GW signal is contained in the ISC.)
3. Write the simulation to disk, read it with PyTDI's `Data` interface and compute the second-generation Michelson variable $X_2$ as demonstrated in the example notebook. Do the models match the estimated ASDs again?

In [ ]:
fs = 4 # sampling rate of the data (Hz)
size = (3600 * 24) * fs # one day of simulation in number of samples
oms_asds = (6.35e-12, 1.25e-11, 1.42e-12, 3.38e-12, 3.32e-12, 7.9e-12)

instru = Instrument(
    size=size,
    dt=1 / fs,
    orbits="static",
    lock="six",
    oms_asds=oms_asds,
    testmass_asds=2.4e-15, # TM noise level in m / s^2 / rtHz
)

instru.disable_all_noises(excluding=["oms", "test-mass", "laser"])

instru.export_hdf5("data/lisa.h5", overwrite=True)

In [ ]:
from pytdi import Data
from pytdi.michelson import X2

In [ ]:
data = Data.from_instrument("data/lisa.h5")

In [ ]:
freqs, psd_sci = si.welch(data.measurements["sci_12"], **welch_kwargs)
freqs, psd_ref = si.welch(data.measurements["ref_12"], **welch_kwargs)
freqs, psd_tmi = si.welch(data.measurements["tmi_12"], **welch_kwargs)

plt.loglog(freqs, np.sqrt(psd_sci), label="sci data")
plt.loglog(freqs[1:], asd_oms(freqs[1:], level=oms_asds[0]), label="sci model")

plt.loglog(freqs, np.sqrt(psd_ref), label="ref data")
plt.loglog(freqs[1:], asd_oms(freqs[1:], level=oms_asds[4]), label="ref model")

plt.loglog(freqs, np.sqrt(psd_tmi), label="tmi data")
plt.loglog(freqs[1:], np.sqrt(4 * asd_tm(freqs[1:])**2 + asd_oms(freqs[1:], level=oms_asds[2])**2), label="tmi model")


plt.xlabel("Frequency (Hz)")
plt.ylabel("ASD (Hz/rtHz)")
plt.legend()

In [ ]:
x2 = X2.build(**data.args)(data.measurements)

In [ ]:
def tf_tdi_oms(f, d=8.332):
    omega = 2 * np.pi * f
    return np.abs(4 * np.sin(omega * d) * np.sin(2 * omega * d))

def tf_tdi_tm(f, d=8.332):
    omega = 2 * np.pi * f
    return np.abs(4 * np.sin(omega * d) * np.sin(2 * omega * d)) * np.sqrt(3 + np.cos(2 * omega * d))

In [ ]:
freqs, psd = si.welch(x2[BUFFER:-BUFFER], **welch_kwargs)

plt.loglog(freqs, np.sqrt(psd), label=r"$X_2$ data")
plt.loglog(freqs[1:], 2 * tf_tdi_oms(freqs[1:]) * asd_oms(freqs[1:]), label="OMSabove model")
plt.loglog(freqs[1:], 2 * tf_tdi_tm(freqs[1:]) * asd_tm(freqs[1:]), label="TM model")

plt.ylim(1e-10, 1e-3)

plt.xlabel("Frequency (Hz)")
plt.ylabel("ASD (Hz/rtHz)")
plt.legend()